# PIT channel search per YOLO26 — pipeline completa (Colab)

**Ricerca PIT (PLiNIO) → export del modello prunato → validazione del modello esportato → fine-tuning → validazione finale.**
Niente export ONNX.

Come usarlo:
1. `Runtime → Cambia tipo di runtime → GPU` (va anche su CPU, ma molto lento).
2. Carica lo zip `pit_yolo_colab.zip` (contiene `pit_yolo.py`, `pit_block_masks.py`, `search.yaml` e questo notebook):
   - **da Google Drive**: metti il percorso dello zip in `ZIP_ON_DRIVE` nella cella 1;
   - **oppure** lascia `ZIP_ON_DRIVE` vuoto e ti verrà chiesto di caricarlo.
3. Imposta i parametri nella cella **3 · Configurazione** (modello, dataset, N, epoche, scheduler…).
4. `Runtime → Esegui tutto`.

Versioni testate: PLiNIO commit `3d6b5e0`, Ultralytics 8.4.165, PyTorch 2.14. Colab ha già PyTorch installato (la versione può essere diversa): la cella 2 lo stampa.

## 1 · File della pipeline

In [ ]:
import os, sys, zipfile, glob, shutil

ZIP_ON_DRIVE = ""                 # es. "/content/drive/MyDrive/pit_yolo_colab.zip" ("" = upload manuale)
WORKDIR = "/content/pit_yolo"     # cartella di lavoro su Colab

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

NEEDED = ("pit_yolo.py", "pit_block_masks.py")
if all(os.path.exists(f) for f in NEEDED):                  # notebook aperto dalla cartella del progetto
    WORKDIR = os.getcwd()
elif not all(os.path.exists(os.path.join(WORKDIR, f)) for f in NEEDED):
    os.makedirs(WORKDIR, exist_ok=True)
    if ZIP_ON_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        zpath = ZIP_ON_DRIVE
    else:
        from google.colab import files
        up = files.upload()                                   # scegli pit_yolo_colab.zip
        zpath = next(k for k in up if k.endswith(".zip"))
    with zipfile.ZipFile(zpath) as z:
        z.extractall(WORKDIR)
    for f in NEEDED:                                          # zip con o senza cartella interna
        if not os.path.exists(os.path.join(WORKDIR, f)):
            src = glob.glob(os.path.join(WORKDIR, "**", f), recursive=True)[0]
            shutil.copy(src, WORKDIR)
            yml = os.path.join(os.path.dirname(src), "search.yaml")
            if os.path.exists(yml):
                shutil.copy(yml, WORKDIR)

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print("cartella di lavoro:", WORKDIR, "| file:", sorted(os.listdir(WORKDIR)))

: 

## 2 · Dipendenze
Installa Ultralytics (versione testata), le dipendenze di PLiNIO e PLiNIO stesso al commit testato. Se sono già presenti non reinstalla nulla.

In [ ]:
import importlib, subprocess

ULTRALYTICS_VERSION = "8.4.165"
PLINIO_COMMIT = "3d6b5e08b3c7d861620ecc9ab25bff82852f8c7c"

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

def installed(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

try:
    import ultralytics
    ok = ultralytics.__version__ == ULTRALYTICS_VERSION
except ImportError:
    ok = False
if not ok:
    pip(f"ultralytics=={ULTRALYTICS_VERSION}")
for mod, pkg in (("networkx", "networkx"), ("tdigest", "tdigest"), ("onnx", "onnx"), ("yaml", "pyyaml")):
    if not installed(mod):
        pip(pkg)
if not installed("plinio"):
    if not os.path.exists("/content/plinio"):
        subprocess.run(["git", "clone", "-q", "https://github.com/eml-eda/plinio.git", "/content/plinio"], check=True)
    subprocess.run(["git", "-C", "/content/plinio", "checkout", "-q", PLINIO_COMMIT], check=True)
    pip("--no-deps", "/content/plinio")
importlib.invalidate_caches()

import torch, ultralytics, plinio
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"ultralytics {ultralytics.__version__} | plinio da {os.path.dirname(plinio.__file__)}")
if ultralytics.__version__ != ULTRALYTICS_VERSION:
    print(f"ATTENZIONE: testato con ultralytics {ULTRALYTICS_VERSION} (hai {ultralytics.__version__}): "
          f"se hai già importato una versione diversa, riavvia il runtime e riesegui")

## 3 · Configurazione
L'unica cella da modificare.

In [ ]:
# ---------------------------------------------------------------- modello e dati
MODEL = "yolo26n.pt"        # il TUO modello già addestrato sui dati (es. "/content/drive/MyDrive/best.pt")
DATA = "coco128.yaml"       # yaml del dataset (es. "/content/drive/MyDrive/aod4/data.yaml")
N = 8                       # canali potati a blocchi di N: sceglilo sulla larghezza SIMD del target
COST = "ops"                # "ops" (MAC) oppure "params"

# ---------------------------------------------------------------- training
SEARCH_EPOCHS = 30
FT_EPOCHS = 15
IMGSZ = 640
BATCH = 16
NBS = 64                    # batch nominale di Ultralytics (accumula i gradienti fino a NBS immagini)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 2                 # Colab ha 2 core

# ---------------------------------------------------------------- ricerca PIT
LAM = 1.0                   # peso del termine di costo
NAS_OPTIMIZER = "AdamW"     # maschere: AdamW | Adam | SGD
NAS_LR0 = 0.01              # scheduler maschere: stesse formule di lr0/lrf/cos_lr di Ultralytics,
NAS_LRF = 0.01              #   calcolate sulle epoche di ricerca; NAS_LRF = 1.0 -> costante
NAS_COS_LR = False
PIT_WARMUP_EPOCHS = 0       # epoche con maschere ferme prima della ricerca (warning se > 0)

# ---------------------------------------------------------------- pesi
OPTIMIZER = "SGD"           # qualsiasi optimizer di Ultralytics, anche "auto": le maschere non ci entrano mai
LR0 = 1e-3
LRF_SEARCH = 1.0            # lr dei pesi durante la ricerca: 1.0 -> costante
LRF_FT = 0.01               # lr dei pesi nel fine-tuning: lineare da LR0 a LR0*LRF_FT

# ---------------------------------------------------------------- alternativa: tutto da YAML
SEARCH_YAML = None          # es. "search.yaml": se impostato, la ricerca usa quel file
                            #   (DATA/SEARCH_EPOCHS/... sopra valgono solo per il resto della pipeline)

# ---------------------------------------------------------------- output
PROJECT = "/content/pit_runs"   # metti un percorso su Drive per non perdere i risultati
NAME = f"n{N}"
PLOTS = True

## 4 · Modello di partenza
Validazione di riferimento del modello in ingresso.

In [ ]:
import csv, json, time
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from ultralytics import YOLO
from pit_yolo import PITYOLO, PrunedTrainer

RUN = Path(PROJECT) / NAME
COMMON = dict(imgsz=IMGSZ, device=DEVICE, workers=WORKERS, project=str(RUN), exist_ok=True)
summary, t0 = {"config": {k: v for k, v in globals().items() if k.isupper() and
                          isinstance(v, (int, float, str, bool, type(None)))}}, time.time()

ref = YOLO(MODEL).val(data=DATA, batch=BATCH, name="val_input_model", plots=False, **COMMON)
summary["map50_95_input"] = float(ref.box.map)
summary["params_input"] = sum(p.numel() for p in YOLO(MODEL).model.parameters())
print(f"modello di partenza: mAP50-95 {ref.box.map:.4f}, parametri {summary['params_input']:,}")

## 5 · Ricerca PIT
Pesi e maschere hanno optimizer e scheduler separati; AMP sempre spenta. Output nella cartella `search/` del run: tutto quello di Ultralytics più `pit_results.png`, `channels.csv`, `pit_args.yaml`.

In [ ]:
search = PITYOLO(MODEL, n=N, cost=COST, trace_imgsz=IMGSZ)
if SEARCH_YAML:
    metrics = search.train(cfg=SEARCH_YAML, name="search", project=str(RUN), exist_ok=True)
else:
    metrics = search.train(data=DATA, epochs=SEARCH_EPOCHS, batch=BATCH, nbs=NBS, lam=LAM,
                           nas_optimizer=NAS_OPTIMIZER, nas_lr0=NAS_LR0, nas_lrf=NAS_LRF,
                           nas_cos_lr=NAS_COS_LR, pit_warmup_epochs=PIT_WARMUP_EPOCHS,
                           optimizer=OPTIMIZER, lr0=LR0, lrf=LRF_SEARCH, name="search",
                           plots=PLOTS, **COMMON)
tr = search.trainer
summary["map50_95_end_of_search"] = float(metrics["metrics/mAP50-95(B)"])
summary["real_cost_fraction"] = float(search.model.real_cost_fraction())

# separazione degli optimizer: nessun parametro in comune
w_ids = {id(p) for g in tr.optimizer.param_groups for p in g["params"]}
n_ids = {id(p) for g in tr.nas_optimizer.param_groups for p in g["params"]}
assert w_ids.isdisjoint(n_ids), "maschere finite nell'optimizer dei pesi!"
print(f"optimizer pesi {type(tr.optimizer).__name__} ({len(w_ids)} parametri) e maschere "
      f"{type(tr.nas_optimizer).__name__} ({len(n_ids)} parametri): separati")
print(f"fine ricerca: mAP50-95 {summary['map50_95_end_of_search']:.4f}, "
      f"costo reale {summary['real_cost_fraction']:.3f} dell'originale")
if (Path(tr.save_dir) / "pit_results.png").exists():
    display(Image(filename=str(Path(tr.save_dir) / "pit_results.png"), width=900))

## 6 · Export del modello prunato e validazione
Il modello esportato è un normale modello PyTorch con meno canali (valori delle BN addestrati ripristinati). La validazione usa lo stesso batch del trainer (2 × `BATCH`): con `rect` batch diversi danno mAP leggermente diversi anche a modello identico.

In [ ]:
pruned = search.export_pruned()          # <run>/search/weights/pruned.pt
m0 = pruned.val(data=DATA, batch=BATCH * 2, name="val_after_export", plots=PLOTS, **COMMON)
summary["map50_95_after_export"] = float(m0.box.map)
summary["params_pruned"] = sum(p.numel() for p in pruned.model.parameters())
diff = abs(m0.box.map - summary["map50_95_end_of_search"])
print(f"mAP50-95 fine ricerca {summary['map50_95_end_of_search']:.4f} | modello esportato "
      f"{m0.box.map:.4f} | differenza {diff:.4f} -> {'OK' if diff < 1e-3 else 'DA CONTROLLARE'}")
print(f"parametri {summary['params_input']:,} -> {summary['params_pruned']:,}")

rows = search.model.channel_report()
ch = pd.DataFrame(rows, columns=["layer", "canali", "tenuti", "prunabile"])
changed = ch[ch.canali != ch.tenuti]
print(f"canali prunabili {ch[ch.prunabile].canali.sum()} -> {ch[ch.prunabile].tenuti.sum()} "
      f"({len(changed)} layer modificati), tutti multipli di {N}: "
      f"{bool((changed.tenuti % N == 0).all())}")
display(changed if len(changed) else "nessun canale rimosso")

## 7 · Fine-tuning del modello prunato

In [ ]:
pruned.train(data=DATA, epochs=FT_EPOCHS, batch=BATCH, nbs=NBS, optimizer=OPTIMIZER, lr0=LR0,
             lrf=LRF_FT, name="finetune", trainer=PrunedTrainer, plots=PLOTS, **COMMON)
ft_dir = RUN / "finetune"
if (ft_dir / "results.png").exists():
    display(Image(filename=str(ft_dir / "results.png"), width=900))

## 8 · Validazione finale e riepilogo

In [ ]:
m1 = pruned.val(data=DATA, batch=BATCH, name="val_final", plots=PLOTS, **COMMON)
summary["map50_95_after_finetune"] = float(m1.box.map)
summary["weights_finetuned"] = str(ft_dir / "weights" / "best.pt")
summary["minutes"] = round((time.time() - t0) / 60, 1)
(RUN / "summary.json").write_text(json.dumps(summary, indent=2, default=str))

display(pd.DataFrame([
    ("modello di partenza", summary["map50_95_input"], summary["params_input"], 1.0),
    ("fine ricerca", summary["map50_95_end_of_search"], None, summary["real_cost_fraction"]),
    ("prunato, dopo export", summary["map50_95_after_export"], summary["params_pruned"],
     summary["real_cost_fraction"]),
    ("prunato, dopo fine-tuning", summary["map50_95_after_finetune"], summary["params_pruned"],
     summary["real_cost_fraction"]),
], columns=["fase", "mAP50-95", "parametri", "costo reale"]).astype({"parametri": "Int64"}))
print(f"riepilogo salvato in {RUN / 'summary.json'} | modello finale: {summary['weights_finetuned']}")